# 🏥 DataProjectLab — Projet Santé
## Prévision de la demande en médicaments
### Notebook 3 — Machine Learning & Prévisions (Module Premium)
---
**Prérequis :** Notebook 2 complété + `df_features` disponible avec le feature engineering.

## 🎯 Objectif ML

Tu vas construire un système de prévision capable de **prédire la consommation hebdomadaire** de chaque médicament pour les 4 prochaines semaines.

Cette prévision permettra au Dr. Konan de savoir **quand et combien commander** pour chaque médicament, en tenant compte des délais fournisseurs.

## Approche retenue : deux modèles complémentaires

| Modèle | Usage | Avantage |
|---|---|---|
| **Random Forest Regressor** | Prévision par médicament avec features | Capture les patterns complexes |
| **Prophet (Meta)** | Décomposition tendance + saisonnalité | Interprétable, gère les jours fériés |

Tu vas entraîner les deux et comparer leurs performances sur un jeu de test.

In [ ]:
# Imports ML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Charge le dataset features (issu du Notebook 2)
# df_features = pd.read_csv('df_features.csv', parse_dates=['date'])
# OU recharge depuis les CSV si tu n'as pas sauvegardé

print("✅ Librairies chargées")
print("💡 Rappel : pip install prophet si non installé (pour la partie B)")


---
## PARTIE A — RANDOM FOREST PAR MÉDICAMENT

### 🤖 Tâche 1 : Stratégie de validation temporelle

**⚠️ Règle d'or :** En séries temporelles, on ne fait **jamais** de validation croisée classique (on ne peut pas entraîner sur le futur pour prédire le passé).

**Stratégie à utiliser :**
- Train : Janvier 2022 → Décembre 2023 (24 mois)
- Test : Janvier 2024 → Juin 2024 (6 mois)

**Question avant de coder :** Pourquoi cette règle est-elle importante ? Que se passerait-il si tu utilisais un `train_test_split` aléatoire ?

In [ ]:
# ✏️ TÂCHE 1 — Séparation train/test temporelle

# Définis les dates de coupure
DATE_TRAIN_END = '2023-12-31'
DATE_TEST_START = '2024-01-01'

# Pour un médicament (ex: MED008 - Insuline, le plus critique)
MED_CIBLE = 'MED008'

# df_med8 = df_features[df_features['id_medicament'] == MED_CIBLE].copy()

# 👉 CRÉE df_train ET df_test à partir de df_med8

# Colonnes features (exclure date, id, target)
FEATURE_COLS = ['consommation_7j','consommation_30j','lag_7','lag_28',
                'mois','jour_semaine','est_weekend','semaine_annee']
TARGET = 'quantite_consommee'

# X_train, y_train = ...
# X_test, y_test   = ...

# print(f"Train : {len(X_train)} jours | Test : {len(X_test)} jours")


---
### 🤖 Tâche 2 : Entraînement et évaluation Random Forest

**Métriques à calculer :**
- **MAE** (Mean Absolute Error) — erreur moyenne en unités → le plus interprétable pour le pharmacien
- **RMSE** (Root Mean Squared Error) — pénalise les grandes erreurs
- **MAPE** (Mean Absolute Percentage Error) — erreur en % → facilement communicable
- **R²** — qualité globale du fit

> 💡 **Interprétation pour le business :** Un MAE de 3 unités sur l'Insuline (MED008) signifie que ta prévision se trompe en moyenne de 3 injections par jour. Est-ce acceptable ?

In [ ]:
# ✏️ TÂCHE 2 — Entraînement Random Forest

# 1. Instancie et entraîne le modèle
# rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
# rf.fit(X_train, y_train)

# 2. Prédictions sur le test
# y_pred = rf.predict(X_test)

# 3. Calcule les métriques
def evaluate_model(y_true, y_pred, model_name="Modèle"):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'='*40}")
    print(f"  {model_name}")
    print(f"{'='*40}")
    print(f"  MAE  : {mae:.2f} unités")
    print(f"  RMSE : {rmse:.2f} unités")
    print(f"  MAPE : {mape:.1f} %")
    print(f"  R²   : {r2:.3f}")
    return {'MAE':mae,'RMSE':rmse,'MAPE':mape,'R2':r2}

# 👉 APPELLE evaluate_model avec tes prédictions
# scores_rf = evaluate_model(y_test, y_pred, "Random Forest — MED008 Insuline")


---
### 🤖 Tâche 3 : Feature Importance & Visualisation

**Objectif :** Comprendre quelles variables expliquent le plus la consommation.

La feature importance du Random Forest te dit : *"Quelle part de la variance ma forêt explique grâce à cette variable ?"*

> 💡 **Question business :** Si `lag_7` est la variable la plus importante, qu'est-ce que cela signifie pour la pharmacie ? Quel avantage opérationnel en tire-t-on ?

In [ ]:
# ✏️ TÂCHE 3 — Feature Importance

# 1. Extraire les importances
# importances = pd.DataFrame({
#     'feature': FEATURE_COLS,
#     'importance': rf.feature_importances_
# }).sort_values('importance', ascending=True)

# 2. Visualiser (barh chart horizontal)
# fig, ax = plt.subplots(figsize=(8, 5))
# ax.barh(importances['feature'], importances['importance'])
# ax.set_title("Importance des variables — Random Forest (MED008 Insuline)")
# plt.tight_layout()
# plt.show()

# 3. Visualiser prédictions vs réel
# fig, ax = plt.subplots(figsize=(14, 5))
# 👉 ÉCRIS TON CODE ICI


---
## PARTIE B — PROPHET (DÉCOMPOSITION SAISONNIÈRE)

Prophet est un outil développé par Meta, idéal pour les séries temporelles avec tendances et saisonnalités. Il est plus interprétable qu'un Random Forest car il décompose explicitement : **tendance + saisonnalité hebdomadaire + saisonnalité annuelle**.

```bash
pip install prophet
```

In [ ]:
# ✏️ TÂCHE 4 — Modèle Prophet

# from prophet import Prophet

# Format requis par Prophet : colonnes 'ds' (date) et 'y' (valeur)
# df_prophet = df_med8[['date','quantite_consommee']].rename(
#     columns={'date':'ds','quantite_consommee':'y'}
# )

# 1. Instancie Prophet avec saisonnalités
# model = Prophet(
#     yearly_seasonality=True,
#     weekly_seasonality=True,
#     daily_seasonality=False,
#     changepoint_prior_scale=0.05  # rigidité de la tendance
# )

# 2. Entraîne sur les données train
# df_train_prophet = df_prophet[df_prophet['ds'] <= DATE_TRAIN_END]
# model.fit(df_train_prophet)

# 3. Génère les prévisions sur 6 mois + 4 semaines futures
# future = model.make_future_dataframe(periods=6*30 + 28)
# forecast = model.predict(future)

# 4. Visualise la décomposition
# fig = model.plot_components(forecast)
# plt.suptitle('Décomposition Prophet — MED008 Insuline', y=1.02)
# plt.show()


---
### 🤖 Tâche 5 : Comparaison des modèles & Sélection

**Objectif :** Choisir le meilleur modèle pour chaque médicament et justifier ce choix.

Tous les médicaments ne se comportent pas pareil :
- Un médicament avec forte saisonnalité → Prophet sera meilleur
- Un médicament avec patterns complexes → Random Forest sera meilleur

> 💡 **Conseil :** Pour la présentation au Dr. Konan, utilise Prophet pour les visualisations (plus lisible pour un non-data) et Random Forest pour les prévisions opérationnelles (plus précis).

In [ ]:
# ✏️ TÂCHE 5 — Comparaison sur 3 médicaments critiques

meds_a_tester = {
    'MED008': 'Insuline Glargine',
    'MED002': 'Paracétamol 1g',
    'MED011': 'Ceftriaxone 1g'
}

resultats = {}

for med_id, med_nom in meds_a_tester.items():
    print(f"\n🔄 Traitement : {med_nom}")
    
    # 👉 ENTRAÎNE RF ET PROPHET SUR CE MÉDICAMENT
    # Stocke les scores dans resultats[med_id]
    pass

# Affiche le tableau comparatif
# df_resultats = pd.DataFrame(resultats).T
# print(df_resultats.round(3))


---
## PARTIE C — SYSTÈME DE RECOMMANDATION DE COMMANDE

C'est la partie la plus utile pour le Dr. Konan : transformer les prévisions en **quantités à commander**.

### Formule de stock de sécurité optimisé :

```
Stock_sécurité = Z × σ_demande × √(délai_livraison)

Où :
  Z = 1.65 (pour un taux de service à 95%)
  σ_demande = écart-type de la consommation journalière
  délai_livraison = délai du fournisseur en jours
```

### Point de commande (Reorder Point) :
```
ROP = (consommation_journalière_moyenne × délai_livraison) + stock_sécurité
```

In [ ]:
# ✏️ TÂCHE 6 — Calcul du système de commande optimal

def calcul_stock_securite(df_conso_med, delai_frn, z=1.65):
    """
    Calcule le stock de sécurité et le point de commande pour un médicament.
    
    Paramètres:
        df_conso_med : données de consommation pour 1 médicament
        delai_frn    : délai de livraison du fournisseur en jours
        z            : facteur de service (1.65 = 95%)
    
    Retourne:
        dict avec stock_securite, point_commande, quantite_commande_recommandee
    """
    # 👉 IMPLÉMENTE LA FORMULE ICI
    
    conso_moy_jour = None  # moyenne quotidienne
    sigma_conso    = None  # écart-type
    stock_secu     = None  # formule ci-dessus
    rop            = None  # reorder point
    
    return {
        'conso_moy_jour': conso_moy_jour,
        'sigma_conso': sigma_conso,
        'stock_securite_recommande': stock_secu,
        'point_commande': rop
    }

# Applique à tous les médicaments critiques et génère le tableau de bord de commande
# 👉 ÉCRIS TON CODE ICI


---
## PARTIE D — PRÉVISIONS 4 SEMAINES (LIVRABLE FINAL)

Cette partie génère le fichier final que tu exporteras dans Power BI.

In [ ]:
# ✏️ TÂCHE 7 — Export des prévisions 4 semaines

# Génère les prévisions pour les 4 prochaines semaines (28 jours)
# pour chaque médicament critique

# Structure du fichier de sortie :
# id_medicament | nom | semaine | quantite_prevue | borne_basse | borne_haute | stock_securite | a_commander

# 👉 ÉCRIS TON CODE ICI

# Sauvegarde
# df_previsions.to_csv('previsions_4semaines.csv', index=False)
# print("✅ Fichier previsions_4semaines.csv généré")
# print(df_previsions.head(20))


---
## ✅ Checklist Notebook 3 (ML)

- [ ] Tâche 1 — Séparation train/test temporelle correcte (pas de leakage)
- [ ] Tâche 2 — Random Forest entraîné + 4 métriques calculées
- [ ] Tâche 3 — Feature importance visualisée + interprétée business
- [ ] Tâche 4 — Modèle Prophet entraîné + décomposition visualisée
- [ ] Tâche 5 — Comparaison RF vs Prophet sur 3 médicaments
- [ ] Tâche 6 — Formule stock de sécurité implémentée pour tous les médicaments critiques
- [ ] Tâche 7 — Fichier `previsions_4semaines.csv` exporté

> ✅ Tout complété ? Passe au **Guide Power BI** pour construire le dashboard final.